In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import mode
import scanpy as sc
import warnings
from matplotlib import pyplot as plt
import anndata as ad

from typing import Dict, Optional, Union
import torch
from tqdm import tqdm
from scipy.sparse import csr_matrix
import scib

sys.path.insert(0, "/lustre/scratch126/cellgen/lotfollahi/zh4/scGPT-spatial")
import scgpt_spatial
warnings.filterwarnings("ignore", category=ResourceWarning)

In [2]:
base_dir = Path("/lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/Zero_Shot_Inference/QC_Data")  

adatas = []
qc_files = sorted(base_dir.glob("*QC_norm.h5ad")) 

for f in qc_files:
    adata = sc.read_h5ad(f)
    adata.var_names = adata.var_names.astype(str)
    adata.var_names_make_unique()
        
    adatas.append(adata)

/lustre/scratch126/cellgen/lotfollahi/zh4/Environments/scGPT_2/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/lustre/scratch126/cellgen/lotfollahi/zh4/Environments/scGPT_2/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/lustre/scratch126/cellgen/lotfollahi/zh4/Environments/scGPT_2/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/lustre/scratch126/cellgen/lotfollahi/zh4/Environments/scGPT_2/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_

In [3]:
# Concat all QC-ed zero-shot embryo data
adata_merged = ad.concat(adatas, join='inner', merge='unique')

print(adata_merged.shape)

(60401, 18085)


In [4]:
# Filter out cells where 'C_scANVI' is 'NA' or NaN
adata_merged = adata_merged[~adata_merged.obs['C_scANVI'].isna() & (adata_merged.obs['C_scANVI'] != 'NA')]

print(f"After filtering: {adata_merged.shape[0]} cells remaining")

After filtering: 51937 cells remaining


In [5]:
model_dir = '/lustre/scratch126/cellgen/lotfollahi/zh4/Codebase_of_Zichen/Zichen_NicheJEPA/Embryo_Figure/Generate_embeddings/scGPTspatial/scGPT_spatial_model'
gene_col = 'index'
batch_id_col = 'sample'
cell_type_col = 'C_scANVI'

In [6]:
scGPT_adata_merged = scgpt_spatial.tasks.embed_data(
    adata_merged,
    model_dir,
    gene_col=gene_col,
    obs_to_save=cell_type_col, 
    batch_size=64,
    return_new_adata=True,
)

/lustre/scratch126/cellgen/lotfollahi/zh4/scGPT-spatial/scgpt_spatial/tasks/cell_emb.py:264: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["index"] = adata.var.index
/lustre/scratch126/cellgen/lotfollahi/zh4/scGPT-spatial/scgpt_spatial/tasks/cell_emb.py:134: RuntimeWarning: Mean of empty slice.
  self.gene_stats_dict.loc[i] = [float(values.mean())]
/lustre/scratch126/cellgen/lotfollahi/zh4/Environments/scGPT_2/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/lustre/scratch126/cellgen/lotfollahi/zh4/scGPT-spatial/scgpt_spatial/tasks/cell_emb.py:134: RuntimeWarning: Mean of empty slice.
  self.gene_stats_dict.loc[i] = [float(values.mean())]
/lustre/scratch126/cellgen/lotfollahi/zh4/Environments/scGPT_2/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtyp

In [7]:
scGPT_adata_merged.obs['sample'] = adata_merged.obs['sample']

In [8]:
Path = "/lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/Zero_Shot_Inference/Embedding_data"
scGPT_adata_merged.write_h5ad(Path + "/scGPT_spatial_embeds.h5ad")